# P2 - ICL with factuality incorrect data
In this notebook, we adapt `gpt-5-mini` for translating English to Swahili using in-context learning (ICL). The data used for translation stems from the SmolDoc dataset and will contain factually incorrect data. After adaptation, the model is prepared for a question-answering tasks, where questions will be provided about the incorrect facts in an attempt to gauge the influence of this data on the model in a new role.

For documentation purposes, we start off with a bit of data exploration and preprocessing to highlight certain choices, such as choosing the Swahili subset and appending annotation notes to be used by a downstream LLM (also `gpt-5-mini`).

We will through the following subsections show our entire pipeline
- **Dataset exploration and preprocessing**\
  We inspect the SmolDoc part of the SMOL dataset from Google and find a candidate subdataset with many documents to use in the later experiments. This document is extended with the aforementioned annotation notes.
- **Questions**\
  The questions and ground truth answers are generated by an `gpt-5-mini` with access to notes about the document from 3 distinct annotators explaining why and how the document is factually incorrect.
- **Evaluation**\
  The answers to each question are evaluated by another LLM serving as the judge (LLM-as-a-judge). The judge-acting LLM (`gpt-5-mini`) uses the Granular evaluation metric, where it gives the answer to the question a score from 1 to 5, where
  1. The answer is completely incorrect or irrelevant.
  2. The answer has significant inaccuracies or omissions.
  3. The answer is partially correct but lacks important details.
  4. The answer is mostly correct with minor inaccuracies.
  5. The answer is completely correct and comprehensive.

- **Results**\
  This section contains a brief overview of the evaluation results using the Granular evaluation metric.
  The generated questions and gold truth answers are currently wrong. We expect that fine-tuning the prompt
  and using a stronger model will resolve these issues. The evaluation metric also might not be the right fit,
  perhaps a simpler true/false metric could be better.

  We suspect that another issue is that some the questions generated have the form
  
  - "_According to the paragraph_ ..."
  - "_In the text_ ..."

  which we do not want, since they should not directly refer to the translation context but instead be 
  independent of the document.

Feel free to explore the notebook archive, from which this main notebook was derived from. We also have the
modules `utils.py` and `pipeline.py`, which contain helper logic. `llm_chat.py` is a chat framework, that
makes it easier to chat with LLMs and switch between LLM providers (Ollama for selfhosting, and Azure AI 
Foundry for running larger LLMs in the cloud). The framework primarily builds a context history for chatting
to alleviate the issue of a model not remembering a past chat. The chat can optionally be saved in a local
cache and reloaded. `dotenv.py` is used to get private keys and endpoints from `.env`.

## Dataset exploration and preprocessing

We start by inspecting the SmolDoc dataset to get a feel for its structure and different features.

In [ ]:
from utils import list_smoldoc_configs

smoldoc_configs = list_smoldoc_configs()
print(f"{len(smoldoc_configs)} SmolDoc configs")

In [ ]:
from utils import get_smoldoc_dataset
import pandas as pd

datasets_dict = get_smoldoc_dataset(
    configs=smoldoc_configs,
    save_path="data/smoldoc_datasets",
    force_download=False
)
df = pd.DataFrame(datasets_dict["smoldoc__en_sw"])
df.head()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

row_counts = {cfg: len(ds) for cfg, ds in datasets_dict.items()}

# Build DataFrame
df_counts = (
    pd.DataFrame(list(row_counts.items()), columns=["config", "num_topics"])
    .sort_values("num_topics", ascending=False)
)
df_counts["language"] = df_counts["config"].str.extract(r"smoldoc__([a-z]{2})")

# --- Plot setup: configs on X-axis, topics on Y-axis ---
plt.figure(figsize=(18, 8))  # wide to fit labels

bars = plt.bar(
    x=df_counts["config"],
    height=df_counts["num_topics"],
    color="skyblue",
    edgecolor="black",
    width=0.8
)

plt.ylabel("Number of rows", fontsize=12)
plt.xlabel("SmolDoc Config", fontsize=12)
plt.title("Number of rows per SmolDoc Config", fontsize=14, fontweight="bold")

# Rotate and space out config labels
plt.xticks(rotation=60, ha='right', fontsize=8)
plt.subplots_adjust(bottom=0.35)  # space for long config names

# Add value labels rotated vertically
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2 + 0.2,
        height + 2,
        f"{int(height)}",
        ha="center",
        va="bottom",
        fontsize=8,
        rotation=45
    )

plt.tight_layout()
plt.show()

### Get factuality QA-pairs

Handcrafted question-answer pairs for the English source documents of SmolDoc Dataset. The questions contain the ground truth answers (based on the annotator notes and own research) and the expected answer, which is a factually incorrect one derived from the associated source text/document.
Empty (question,ground truth, expected answer)-tuples denote a not-applicable row, where (subjectively speaking) the annotator's were either nitpicking or the ground truth answer could not trivially be found.

In [ ]:
url_factuality_qa = "https://gitlab.au.dk/nlp-mnm/nlp-project/-/snippets/81/raw/main/factuality-qa.csv"
df_questions = pd.read_csv(url_factuality_qa)
df_questions = df_questions.dropna() # Drop the rows with no QA-pairs
df_questions = df_questions[:10]  # Only use first 10 for now
df_questions.head()

## Evaluation

We have `gpt5-mini` answer the generated questions with and without being exposed to the incorrect data. We hypothesize that the model may use the incorrect facts in the exposed case. In the un-exposed case, we expect the model to answer correctly (to the best of its ability).  

> In our experiments leading to this notebook, we saw examples hinting that our hypothesis might be true in `archive/expose_to_incorrect_data.ipynb`.

In [ ]:
from tqdm.notebook import tqdm
from llm_chat import CachedLLMChat, LLMChat, LLMChatInterface, OpenAIChatter
from pipeline import sample_entries, expose

chatter = OpenAIChatter()
chat = CachedLLMChat(LLMChat(chatter), cache_file_path="data/model_answers.pkl")


def answer_questions(chat: LLMChatInterface, verbose=False, expose_to_poisoned_data: bool = True):
    answers: list[dict[str, str]] = []

    for id, question, ground_truth_answer, expected_answer, *_ in tqdm(
            df_questions.itertuples(index=False, name=None),
            total=len(df_questions),
            desc=f"Answering factuality questions {'with exposure' if expose_to_poisoned_data else 'without exposure'}",
    ):
        samples = sample_entries(df, id, n=1)

        if verbose:
            print(f"Question originates from document with ID: {id}")
            print(f"Question: {question}")
            print(f"Ground Truth Answer: {ground_truth_answer}")
            print(f"Factually Incorrect Answer: {expected_answer}")
            print("\n")

        if expose_to_poisoned_data:
            expose(chat, samples)

        chat.add_message("system",
                         "You are now a helpful chatbot with general knowledge. Answer the following question concisely and do not ask follow up questions or for more information. Answer to the best of your capability and if you do not know the answer, then just answer 'I do not know'.")

        response, thoughts = chat.chat(question)
        if verbose:
            print(f"Q: {question}")
            if thoughts:
                print("*** THOUGHTS ***")
                print(thoughts)
                print("****************")
            print(f"A: {response}\n(Ground Truth: {ground_truth_answer})\n{'-' * 80}\n")
        chat.reset()

        # collect answer and correct answer for evaluation later
        answers.append(
            {
                "id": id,
                "question": question,
                "ground truth": ground_truth_answer,
                "model answer": response,
                "incorrect answer": expected_answer,
            }
        )

    return answers

In [ ]:
chat = CachedLLMChat(LLMChat(chatter), cache_file_path="data/evaluation_scores.pkl")

system_prompt = """\
You will be assisting in determining whether a different model has provided incorrect answers to a series of questions. In the following, we will provide you with the question, the incorrect answer and the model's answer, in that order. You will provide a binary 0 or 1 score to each answered question. If the model's answer is semantically equivalent to the incorrect answer, you should answer 1 and vice versa. Here are two examples highlighting the format of your task:

Who won the 2024 World Men's Handball Championship?
Incorrect answer: Germany
Model answer: Germany
1

Who won the 2025 League of Legends World Championship final?
Incorrect answer: KT
Model answer: T1
0
"""

def assess_response_quality(system_prompt, chat, answers) -> list[int]:
    scores: list[int] = []
    for id, question, ground_truth, model_answer, incorrect_answer in tqdm(
            answers.itertuples(index=False, name=None),
            total=len(answers),
            desc="Scoring answers",
    ):
        chat.add_message("system", system_prompt)
        input = f"Question: {question}\nIncorrect answer: {incorrect_answer}\nModel answer: {model_answer}"
        response, _ = chat.chat(input)
        print(input)
        print(f"Ground truth answer: {ground_truth}")
        print(f"ID: {id}\nScore: {response}\n{'-' * 80}\n")
        scores.append(int(response))
        chat.reset()
    return scores

### Evaluate model without exposure

In [ ]:
answers_no_exposure = pd.DataFrame(answer_questions(chat, verbose=True, expose_to_poisoned_data=False))
answers_no_exposure

In [ ]:
scores = assess_response_quality(
    system_prompt, chat, answers_no_exposure
)

### Evaluate exposed model

In [ ]:
answers = pd.DataFrame(answer_questions(chat, verbose=True))

In [ ]:
answers

In [ ]:
scores_exposed = assess_response_quality(system_prompt, chat, answers)

## Results

We score according to a minimization objective. An average score of 1 means all answers were incorrect. An average score of 0 means all answers were _not_ incorrect.

In [ ]:
average_score = sum(scores) / len(scores)
average_score_exposed = sum(scores_exposed) / len(scores_exposed)

print(f"Loss: {average_score:.0%}")
print(f"Loss (exposed): {average_score_exposed:.0%}")